## Import các thư viện, module và class cần thiết để chương trình Selenium hoạt động

In [18]:
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import pandas as pd
import time

## Tạo hàm nhập dữ liệu giới hạn vào các ô input trên form web

In [4]:
def input_with_limit(field_name, value, limit=50):
    try:
        field = driver.find_element(By.NAME, field_name)
        field.click()
        field.clear()
        if len(value) <= limit:
            field.send_keys(value)
        else:
            driver.execute_script("arguments[0].value = arguments[1];", field, value) # sử dụng javascript 
    except Exception as e:
        print(f"Không nhập được ô {field_name}: {e}") # e là thông báo 

## 1. Kiểm thử tự động phần Đăng nhập

In [22]:
# === 1. Đọc dữ liệu từ file Excel ===
data = pd.read_excel("testdata_automation.xlsx", sheet_name="DangNhap").fillna("")

# === 2. Khởi tạo trình duyệt ===
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
link = "http://127.0.0.1:8000"
driver.get(link)
driver.maximize_window() #phóng to màn hình trình duyệt
time.sleep(3)

results = []

# === 3. Lặp qua từng test case ===
for index, row in data.iterrows():
    username = str(row['Username']).strip()
    password = str(row['Password']).strip()
    expected = str(row['ExpectedResult']).strip()
    testcase_id = row['TestCaseID']

    try:
        wait = WebDriverWait(driver, 10)
        login_button = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//a[contains(@class,'btn-primary')]"))
        )
        login_button.click()
        time.sleep(2)
    except Exception as e:
        print(f"Không tìm thấy nút đăng nhập: {e}")
        continue

    # === Nhập dữ liệu ===
    input_with_limit("username", username)
    input_with_limit("password", password)
    
    driver.find_element(By.NAME, "password").send_keys(Keys.ENTER) 
    time.sleep(2)

    # === 4. Kiểm tra kết quả ===
    try:
        driver.find_element(By.XPATH, "//a[contains(@class,'btn-success')]").click()
        actual = "Success"
        message = "Đăng nhập thành công"
    except:
        actual = "Fail"
        try:
            wait = WebDriverWait(driver, 5)
            error_elements = wait.until(
                EC.presence_of_all_elements_located((By.XPATH, "//li[@style='list-style:none']"))
            )
            message = " | ".join([e.text.strip() for e in error_elements])
        except:
            message = "Không tìm thấy thông báo lỗi"

    # === 5. So sánh kết quả ===
    status = "Passed" if actual.lower() == expected.lower() else "Failed"
    results.append({
        "TestCaseID": testcase_id,
        "Username": username,
        "Password": password,
        "Expected": expected,
        "Actual": actual,
        "Message": message,
        "Status": status
    })

    # Quay lại trang chính để chuẩn bị test tiếp
    driver.get(link)
    time.sleep(2)

# === 6. Đóng trình duyệt ===
driver.quit()

# === 7. Xuất file kết quả ===
pd.DataFrame(results).to_excel("login_test_results.xlsx", index=False)
print("Kiểm thử hoàn tất. Kết quả được lưu tại file login_test_results.xlsx")

Kiểm thử hoàn tất. Kết quả được lưu tại file login_test_results.xlsx


## 2. Kiểm thử tự động phần đăng ký (Cách 1)

In [24]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import NoSuchElementException
from datetime import datetime

def init_driver():
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    driver.get("http://127.0.0.1:8000/")
    time.sleep(0.5)
    return driver

def open_register_page(driver):
    try:
        driver.find_element(By.XPATH, "//a[@class='btn btn-success']").click()
        time.sleep(0.5)
        print("Đã mở trang đăng ký.")
        return True
    except NoSuchElementException:
        print("Không tìm thấy nút đăng ký.")
        return False

def get_date_string(row, testcase_id):
    if testcase_id == "ĐK04":
        return "12/13/2005"
    if testcase_id == "ĐK05":
        return "3/2/2001"

    value = row["Ngày sinh"]
    if pd.isna(value):
        return ""

    if isinstance(value, pd.Timestamp):
        return value.strftime("%d/%m/%Y")

    date_str = str(value).strip().split(" ")[0]
    if "-" in date_str and len(date_str.split("-")[0]) == 4:
        try:
            return pd.to_datetime(date_str).strftime("%d/%m/%Y")
        except:
            return date_str
    return date_str

def fill_form(driver, row, testcase_id):
    fields = {
        "ho": row["Họ"],
        "ten": row["Tên"],
        "email": row["Email"],
        "dienthoai": row["Điện thoại"],
        "username": row["Username"],
        "password": row["Mật khẩu"],
        "password_confirmation": row["Nhập lại mật khẩu"],
        "diachi": row["Địa chỉ"]
    }

    print("Đang điền dữ liệu:")
    for name, value in fields.items():
        field = driver.find_element(By.NAME, name)
        field.clear()
        text = str(value).strip() if pd.notna(value) else ""
        field.send_keys(text)
        print(f" - {name}: {text}")

    try:
        date_field = driver.find_element(By.NAME, "ngaysinh")
        date_field.clear()
        date_str = get_date_string(row, testcase_id)
        date_field.send_keys(date_str)
        print(f" - ngaysinh: {date_str}")
    except NoSuchElementException:
        print("Không tìm thấy field 'ngaysinh'.")

    if pd.notna(row["Giới tính"]):
        Select(driver.find_element(By.NAME, "gioitinh")).select_by_visible_text(row["Giới tính"])
        print(f" - Giới tính: {row['Giới tính']}")
    else:
        print(" - Giới tính: Không chọn")

def submit_form(driver, use_enter=False):
    if use_enter:
        driver.find_element(By.NAME, "password_confirmation").send_keys(Keys.ENTER)
        print("Đã submit bằng phím Enter.")
    else:
        driver.find_element(By.XPATH, "//input[@type='submit']").click()
    time.sleep(0.5)

    try:
        msg_element = driver.find_element(By.XPATH, "//*[contains(@class, 'alert alert-success')]")
        msg = msg_element.text
        actual = "Success" if "đăng ký thành công" in msg.lower() else "Fail"
    except NoSuchElementException:
        try:
            error_element = driver.find_element(By.XPATH, "//span[contains(@class, 'error')]")
            msg = error_element.text
            actual = "Fail"
        except NoSuchElementException:
            try:
                error_alert = driver.find_element(By.XPATH, "//*[contains(@class, 'alert-danger')]")
                msg = error_alert.text
                actual = "Fail"
            except NoSuchElementException:
                msg = "Không có thông báo"
                actual = "No message"

    print(f"Kết quả: {actual} (Message: {msg})")
    return actual, msg

def run_test_case(driver, row, test_id, description, use_enter=False):
    print(f"\nTEST CASE {test_id}: {description}\n{'='*40}")
    if not open_register_page(driver):
        return {
            "TestCaseID": test_id,
            "Description": description,
            "ExpectedResult": row["ExpectedResult"],
            "Actual": "Error",
            "Message": "Không mở được trang đăng ký",
        }

    fill_form(driver, row, test_id)
    actual, msg = submit_form(driver, use_enter)

    driver.get("http://127.0.0.1:8000/")
    time.sleep(0.5)

    return {
        "TestCaseID": test_id,
        "Description": description,
        "ExpectedResult": str(row["ExpectedResult"]).strip(),
        "Actual": actual,
        "Message": msg,
    }

def save_summary_to_excel(results, output_file="ketqua_tomtat.xlsx"):
    try:
        data = []
        for r in results:
            expected = r.get("ExpectedResult", "")
            actual = r.get("Actual", "")
            status = "Pass" if expected.strip().lower() == actual.strip().lower() else "Fail"
            data.append({
                "TestCaseID": r.get("TestCaseID", ""),
                "ExpectedResult": expected,
                "Kết quả sau khi chạy": actual,
                "Trạng thái": status
            })

        df_out = pd.DataFrame(data)
        df_out.to_excel(output_file, index=False, engine="openpyxl")
        print(f" Đã ghi kết quả tóm tắt vào file: {output_file}")
    except Exception as e:
        print(f" Lỗi khi ghi file Excel: {e}")

def main():
    file_path = "testdata_automation.xlsx"  
    try:
        df = pd.read_excel(file_path, sheet_name="DangKy2", dtype=str, engine="openpyxl").apply(
            lambda x: x.str.strip() if x.dtype == "object" else x
        )
        print("Đã đọc file Excel testdata_automation.xlsx thành công.")
    except Exception as e:
        print(f"Lỗi khi đọc Excel: {e}")
        return

    driver = init_driver()
    results = []

    test_cases = {
        "ĐK01": ("Nhập đúng mọi thông tin hợp lệ", False),
        "ĐK04": ("Ngày sinh sai định dạng (MM/dd/YYYY)", False),
        "ĐK05": ("Ngày sinh sai định dạng (d/m/yyyyy)", False),
        "ĐK07": ("Email sai định dạng", False),
        "ĐK08": ("Email đã tồn tại", False),
        "ĐK10": ("Số điện thoại khác 10 số", False),
        "ĐK11": ("Nhập chữ vào số điện thoại", False),
        "ĐK12": ("Username chỉ chứa ký tự số", False),
        "ĐK14": ("Username đã tồn tại", False),
        "ĐK16": ("Mật khẩu không đủ độ phức tạp", False),
        "ĐK17": ("Mật khẩu thiếu ký tự số", False),
        "ĐK18": ("Mật khẩu thiếu ký tự hoa", False),
        "ĐK19": ("Mật khẩu nhập lại không khớp", False),
        "ĐK21": ("Bỏ trống tất cả các trường", False),
        "ĐK23": ("Nhập hợp lệ bằng phím Enter", True),
    }

    for _, row in df.iterrows():
        test_id = row["TestCaseID"]
        if test_id in test_cases:
            desc, use_enter = test_cases[test_id]
            results.append(run_test_case(driver, row, test_id, desc, use_enter))
        else:
            results.append({
                "TestCaseID": test_id,
                "Description": "Không xác định",
                "ExpectedResult": row.get("ExpectedResult", ""),
                "Actual": "Error",
                "Message": "Không có hàm test tương ứng",
            })

    driver.quit()

    output_name = f"ketqua_tomtat_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
    save_summary_to_excel(results, output_name)

    print("\n*** TỔNG KẾT ***")
    for r in results:
        print(f"{r['TestCaseID']}: {r.get('Actual')} - {r.get('Message')}")

if __name__ == "__main__":
    main()


Đã đọc file Excel testdata_automation.xlsx thành công.

TEST CASE ĐK01: Nhập đúng mọi thông tin hợp lệ
Đã mở trang đăng ký.
Đang điền dữ liệu:
 - ho: a
 - ten: a
 - email: nguyenvananhquynhduongduyennguyentrang1234567890abcdefghijklmnopqrstuvwxyzabcdefghijklmno1@gmail.com
 - dienthoai: 0123456789
 - username: name01
 - password: Pass1234
 - password_confirmation: Pass1234
 - diachi: 1234 Đường Nguyễn Văn Cừ, Phường An Khánh, Quận Ninh Kiều, Thành phố Cần Thơ, Việt Nam ABCXYZ 7890PQ
 - ngaysinh: 22/02/2001
 - Giới tính: Nữ
Kết quả: Success (Message: Đăng ký thành công)

TEST CASE ĐK04: Ngày sinh sai định dạng (MM/dd/YYYY)
Đã mở trang đăng ký.
Đang điền dữ liệu:
 - ho: Võ
 - ten: Anh Bình Cường Dũng Giang Hiếu Khang Long An Trúc
 - email: vi@gmail.com
 - dienthoai: 0678678678
 - username: user002
 - password: Hello1235
 - password_confirmation: Hello1235
 - diachi: 
 - ngaysinh: 12/13/2005
 - Giới tính: Nữ
Kết quả: Fail (Message: Vui lòng nhập thông tin Ngày sinh theo định dạng dd/mm/yyy

## Kiểm thử tự động phần Đăng ký (Cách 2)

In [20]:
# === 1. Đọc dữ liệu ===
data = pd.read_excel("testdata_automation.xlsx", sheet_name="DangKy1").fillna("")

# === 2. Khởi tạo trình duyệt ===
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
link = "http://127.0.0.1:8000"
driver.get(link)
driver.maximize_window() #phóng to màn hình trình duyệt
time.sleep(3)

results = []

# === 3. Lặp qua từng test case ===
for index, row in data.iterrows():
    testcase_id = row['TestCaseID']
    last_name = str(row['Họ']).strip()
    first_name = str(row['Tên']).strip()
    gender = str(row['Giới tính']).strip()
    birthday = str(row['Ngày sinh']).strip()
    address = str(row['Địa chỉ']).strip()
    email = str(row['Email']).strip()
    phone = str(row['Điện thoại']).strip()
    username = str(row['Username']).strip()
    password = str(row['Mật khẩu']).strip()
    confirm = str(row['Nhập lại mật khẩu']).strip()
    expected = str(row['ExpectedResult']).strip()

    try:
        wait = WebDriverWait(driver, 10)
        register_button = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//a[contains(@class,'btn-success')]"))
        )
        register_button.click()
        time.sleep(1)
    except Exception as e:
        print(f"Không tìm thấy nút Đăng ký: {e}")
        continue

    # === Nhập dữ liệu ===
    input_with_limit("ho", last_name)
    input_with_limit("ten", first_name)

    try:
        value = "1" if gender.lower() in ["nam", "1"] else "0"
        Select(driver.find_element(By.NAME, "gioitinh")).select_by_value(value)
    except:
        pass

    input_with_limit("ngaysinh", birthday)
    input_with_limit("diachi", address, 100)
    input_with_limit("email", email, 100)
    input_with_limit("dienthoai", phone, 10)
    input_with_limit("username", username)
    input_with_limit("password", password)
    input_with_limit("password_confirmation", confirm)

    driver.find_element(By.NAME, "password_confirmation").send_keys(Keys.ENTER)
    time.sleep(2)

    # === Kiểm tra kết quả ===
    try:
        success_msg = driver.find_element(By.CLASS_NAME, "alert-success")
        message = success_msg.text.strip()
        actual = "Success"
    except:
        try:
            error_elements = driver.find_elements(By.XPATH, "//span[@class='error']")
            if error_elements:
                message = " | ".join([e.text.strip() for e in error_elements])
            else:
                message = "Không tìm thấy thông báo lỗi"
            actual = "Fail"
        except:
            message = "Không tìm thấy thông báo lỗi"
            actual = "Fail"

    status = "Passed" if actual.lower() == expected.lower() else "Failed"
    results.append({
        "TestCaseID": testcase_id,
        "Username": username,
        "Expected": expected,
        "Actual": actual,
        "Message": message,
        "Status": status
    })

    driver.get(link)
    time.sleep(1)

driver.quit()
pd.DataFrame(results).to_excel("register_test_results.xlsx", index=False)
print("Kiểm thử hoàn tất. Kết quả lưu tại file register_test_results.xlsx")

Kiểm thử hoàn tất. Kết quả lưu tại file register_test_results.xlsx
